[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/32_topk_sampling.ipynb)

# 🟠 Medium: Top-k / Top-p (Nucleus) Sampling

Implement **sampling with top-k and top-p filtering** — the standard LLM decoding strategy.

### Signature
```python
def sample_top_k_top_p(logits, top_k=0, top_p=1.0, temperature=1.0) -> int:
    # logits: (V,) unnormalized log-probabilities
    # Returns: sampled token index
```

### Algorithm
1. Scale by temperature: `logits /= temperature`
2. Top-k: keep only top-k logits, set rest to `-inf`
3. Top-p: sort by prob, mask tokens where cumulative prob exceeds p
4. Sample from filtered distribution

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [2]:
import torch

In [17]:
def sample_top_k_top_p(logits, top_k=0, top_p=1.0, temperature=1.0) -> int:
    #scaling by temperature --> temp = 0 means deterministic, temp = 1 means random, temp >> 1 means all tokens with equl prob (useless)
    if temperature == 0.0:
      return torch.argmax(logits, dim = -1)


    sorted_logits, sorted_indices = torch.sort(logits / temperature, dim = -1, descending = True)
    if top_k > 0:
      sorted_logits[..., top_k:] = float("-inf") #... for general case (B, vocab_size)


    if top_p < 1.0:
      probs = torch.softmax(sorted_logits, dim = -1)
      cum_sum = torch.cumsum(probs, dim = -1)
      remove = cum_sum > top_p

      #maintain last token that reaches / exceed top_p
      remove[..., 1:] = remove[..., :-1].clone()
      remove[0] = False #never remove first one even if already exceed top_p

      sorted_logits[..., remove] = float("-inf")


    probs = torch.softmax(sorted_logits, dim = -1)
    chosen_idx = torch.multinomial(probs, num_samples=1)

    #return torch.gather(sorted_indices, dim = -1, index = chosen_idx) --> general case (B, 1)
    return sorted_indices[chosen_idx].item()

In [18]:
# 🧪 Debug
logits = torch.tensor([1.0, 5.0, 2.0, 0.5])
print('top_k=1:', sample_top_k_top_p(logits.clone(), top_k=1))
print('top_p=0.5:', sample_top_k_top_p(logits.clone(), top_p=0.5))
print('temp=0.01:', sample_top_k_top_p(logits.clone(), temperature=0.01))

top_k=1: 1
top_p=0.5: 1
temp=0.01: 1


In [19]:
# ✅ SUBMIT
from torch_judge import check
check('topk_sampling')


🧪 Testing: Top-k / Top-p Sampling (Medium)
──────────────────────────────────────────────────
  ✅ [1/4] top_k=1 always returns argmax (7.1ms)
  ✅ [2/4] Low temperature concentrates (8.2ms)
  ✅ [3/4] All tokens reachable (no filtering) (327.6ms)
  ✅ [4/4] Returns valid index (5.8ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (348.6ms total)
  Progress saved. Run status() to see your dashboard.

